In [ ]:
!pip -q install openai pandas numpy scipy scikit-learn tqdm pillow matplotlib


In [ ]:
from google.colab import drive, userdata
from IPython.display import Markdown, display
from pathlib import Path
from datetime import datetime, timezone
from PIL import Image
import base64
import hashlib
import json
import os
import random
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm.auto import tqdm
from openai import OpenAI

RANDOM_SEED = 20260803
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

                                  
RUN_NEW_API_CALLS = False
RUN_TEST_API_CALLS = False

                                                                            
                                                                      
                                                                                   
PILOT_DEV_ONLY = True
RUN_LIMIT = None

CC_MODEL = 'gpt-5.4'
CC_REASONING_EFFORT = 'none'
CC_IMAGE_DETAIL = 'original'
CC_DEMONSTRATION_IMAGE_DETAIL = 'low'

SCORE_ANCHORS = {
    '0.00': 0.00,
    '0.25': 0.25,
    '0.50': 0.50,
    '0.75': 0.75,
    '1.00': 1.00,
}
BLEND_WEIGHTS = np.round(np.linspace(0.0, 1.0, 21), 2)
OUTER_FOLDS = 5
INNER_FOLDS = 4
MIN_BLEND_GAIN_OVER_VERSION_PRIOR = 0.01
RAW_PILOT_TARGET = 0.635
BOOTSTRAP_REPLICATES = 2000
REQUEST_SLEEP_SECONDS = 0.2
MAX_RETRIES = 3

TARGET_COLUMN = 'CRAI_CC'
GOLD_COLUMN = 'gold_cc'

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 180)

drive.mount('/content/drive')
IMAGEEVAL_ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = IMAGEEVAL_ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_DIR_CANDIDATES = [PROJECT_DIR / 'data', IMAGEEVAL_ROOT / 'train_dev']
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES
     if (path / 'train' / 'captions.tsv').exists()),
    DATA_DIR_CANDIDATES[0],
)

EXPERIMENT_ROOT = PROJECT_DIR / 'cc_qatari_label_anchored_v3'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
for path in [CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

client = None

def get_openai_client():
    global client
    if client is None:
        if not (RUN_NEW_API_CALLS or RUN_TEST_API_CALLS):
            raise RuntimeError(
                'API calls are disabled. Enable the appropriate inference flag explicitly.'
            )
        key = userdata.get('openai')
        if not key:
            raise RuntimeError('Colab secret "openai" is missing.')
        os.environ['OPENAI_API_KEY'] = key
        client = OpenAI()
    return client

print('Data:', DATA_DIR)
print('CC experiment:', EXPERIMENT_ROOT)
print('New API calls enabled:', RUN_NEW_API_CALLS)
print('Pilot dev only:', PILOT_DEV_ONLY)
print('Test API calls enabled:', RUN_TEST_API_CALLS)
print('Judge:', CC_MODEL, '| reasoning:', CC_REASONING_EFFORT)


In [ ]:
ALL_GOLD_COLUMNS = [
    'CRAI_CEA', 'CRAI_CC', 'CRAI_CS', 'CRAI_CI', 'CRAI_HP', 'CRAI_composite'
]

                                                                                
                                                                                     
                                                                                   
                                                                                 
MAPPING_AUDIT_VERSION = 'train-remap-017-019-exclude-020-v1'
ENABLE_VERIFIED_TRAIN_MAPPING_REPAIR = True
VERIFIED_GENERATED_BASE_REMAP = {
    'train': {
        'img_017': 'img_018',
        'img_018': 'img_019',
        'img_019': 'img_020',
    }
}
UNRESOLVED_BASE_IDS = {'train': {'img_020'}}

def parse_image_base_id(instance_id: str) -> str:
    return re.sub(r'_v\d+$', '', str(instance_id))

def parse_caption_version(instance_id: str) -> int:
    match = re.search(r'_v(\d+)$', str(instance_id))
    return int(match.group(1)) if match else -1

def find_existing_image(folder: Path, stem: str) -> Path:
    for extension in ['.png', '.jpg', '.jpeg', '.webp']:
        candidate = folder / f'{stem}{extension}'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'No image found for {stem} in {folder}')

def resolve_generated_source_id(instance_id: str, split: str) -> str:
    instance_id = str(instance_id)
    base_id = parse_image_base_id(instance_id)
    version = parse_caption_version(instance_id)
    remap = VERIFIED_GENERATED_BASE_REMAP.get(split, {}) if (
        ENABLE_VERIFIED_TRAIN_MAPPING_REPAIR
    ) else {}
    source_base = remap.get(base_id, base_id)
    return f'{source_base}_v{version}'

def load_split(split: str, require_gold: bool) -> pd.DataFrame:
    split_dir = DATA_DIR / split
    captions = pd.read_csv(split_dir / 'captions.tsv', sep='\t')
    frame = captions.copy()
    if require_gold:
        gold_path = split_dir / 'gold_human.tsv'
        if not gold_path.exists():
            raise FileNotFoundError(gold_path)
        gold = pd.read_csv(gold_path, sep='\t')
        frame = frame.merge(gold, on='id', how='left', validate='one_to_one')
        if frame[TARGET_COLUMN].isna().any():
            raise ValueError(f'{split}: missing {TARGET_COLUMN} labels')
    frame['id'] = frame['id'].astype(str)
    frame['split'] = split
    frame['base_id'] = frame['id'].map(parse_image_base_id)
    frame['caption_version'] = frame['id'].map(parse_caption_version)
    frame['caption_version_key'] = frame['caption_version'].map(lambda x: f'v{x}')
    if 'category' not in frame.columns:
        frame['category'] = 'unknown'
    frame['category'] = frame['category'].fillna('unknown').astype(str)
    frame['generated_source_id'] = frame['id'].map(
        lambda value: resolve_generated_source_id(value, split)
    )
    frame['mapping_status'] = np.where(
        frame['generated_source_id'].eq(frame['id']), 'identity', 'verified_remap'
    )
    unresolved = UNRESOLVED_BASE_IDS.get(split, set())
    frame['eligible_for_cc'] = ~frame['base_id'].isin(unresolved)
    frame.loc[~frame['eligible_for_cc'], 'mapping_status'] = 'excluded_missing_target'
    frame['ref_image_path'] = frame['base_id'].map(
        lambda value: str(find_existing_image(split_dir / 'imgs' / 'ref', value))
    )
    frame['generated_image_path'] = frame['generated_source_id'].map(
        lambda value: str(find_existing_image(split_dir / 'imgs' / 'generated', value))
    )
    return frame

def assert_group_structure(frame: pd.DataFrame, split: str):
    assert frame['id'].is_unique, f'{split}: duplicate IDs'
    assert frame['base_id'].notna().all(), f'{split}: missing group IDs'
    assert frame.groupby('id')['base_id'].nunique().eq(1).all()
    versions = frame.groupby('base_id')['caption_version'].apply(lambda x: set(map(int, x)))
    bad = versions[versions != {1, 2, 3, 4, 5}]
    assert bad.empty, f'{split}: incomplete caption groups: {bad.to_dict()}'
    assert frame.groupby('base_id')['ref_image_path'].nunique().eq(1).all()
    assert frame['generated_source_id'].is_unique, f'{split}: generated sources reused'

train_all_df = load_split('train', require_gold=True)
dev_all_df = load_split('dev', require_gold=True)
train_df = train_all_df.loc[train_all_df['eligible_for_cc']].reset_index(drop=True)
dev_df = dev_all_df.loc[dev_all_df['eligible_for_cc']].reset_index(drop=True)

assert_group_structure(train_df, 'train')
assert_group_structure(dev_df, 'dev')
assert set(train_df['base_id']).isdisjoint(set(dev_df['base_id']))
assert not set(UNRESOLVED_BASE_IDS.get('train', set())) & set(train_df['base_id'])

TEST_AVAILABLE = (DATA_DIR / 'test' / 'captions.tsv').exists()
test_all_df = load_split('test', require_gold=False) if TEST_AVAILABLE else None
test_df = (
    test_all_df.loc[test_all_df['eligible_for_cc']].reset_index(drop=True)
    if test_all_df is not None else None
)
if test_df is not None:
    assert_group_structure(test_df, 'test')
    assert set(train_df['base_id']).isdisjoint(set(test_df['base_id']))

caption_column_sets = []
for split in ['train', 'dev'] + (['test'] if TEST_AVAILABLE else []):
    caption_column_sets.append(set(pd.read_csv(
        DATA_DIR / split / 'captions.tsv', sep='\t', nrows=1
    ).columns))
USE_CATEGORY_FEATURE = all('category' in columns for columns in caption_column_sets)

def caption_text_from_row(row: pd.Series) -> str:
    for column in ['caption', 'text', 'prompt']:
        if column in row and pd.notna(row[column]):
            return str(row[column])
    raise KeyError('No caption column found')

def make_v1_caption_map(frame: pd.DataFrame) -> dict:
    v1 = frame.loc[frame['caption_version'].eq(1)]
    assert v1['base_id'].is_unique
    return dict(zip(v1['base_id'], v1.apply(caption_text_from_row, axis=1)))

V1_CAPTION_BY_SPLIT = {
    'train': make_v1_caption_map(train_all_df),
    'dev': make_v1_caption_map(dev_all_df),
}
if test_all_df is not None:
    V1_CAPTION_BY_SPLIT['test'] = make_v1_caption_map(test_all_df)

def get_v1_caption(row: pd.Series) -> str:
    return V1_CAPTION_BY_SPLIT[str(row['split'])][str(row['base_id'])]

print('Mapping audit version:', MAPPING_AUDIT_VERSION)
print('train:', len(train_df), 'eligible rows |', train_df['base_id'].nunique(), 'groups')
print('excluded train groups:', sorted(UNRESOLVED_BASE_IDS.get('train', set())))
print('dev:', len(dev_df), 'rows |', dev_df['base_id'].nunique(), 'groups')
print('test available:', TEST_AVAILABLE)
print('Category used as feature:', USE_CATEGORY_FEATURE)
display(train_all_df.loc[
    train_all_df['mapping_status'].ne('identity'),
    ['id', 'generated_source_id', 'mapping_status', TARGET_COLUMN]
].head(25))


In [ ]:
def file_sha256(path: str) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

mapping_manifest = pd.concat([
    train_all_df.assign(in_model=train_all_df['eligible_for_cc']),
    dev_all_df.assign(in_model=dev_all_df['eligible_for_cc']),
] + ([test_all_df.assign(in_model=test_all_df['eligible_for_cc'])]
     if test_all_df is not None else []), ignore_index=True)

mapping_manifest = mapping_manifest[[
    'split', 'id', 'base_id', 'caption_version', 'generated_source_id',
    'mapping_status', 'in_model', 'ref_image_path', 'generated_image_path'
]].copy()
mapping_manifest['generated_sha256'] = mapping_manifest['generated_image_path'].map(
    file_sha256
)

eligible_manifest = mapping_manifest.loc[mapping_manifest['in_model']]
assert not eligible_manifest.duplicated(['split', 'generated_source_id']).any()
mapping_manifest.to_csv(
    OUTPUT_DIR / 'cc_mapping_manifest.tsv', sep='\t', index=False
)

def show_mapping_audit(frame, base_ids=('img_017', 'img_018', 'img_019', 'img_020')):
    selected = frame.loc[
        frame['base_id'].isin(base_ids) & frame['caption_version'].eq(1)
    ].sort_values('base_id')
    for _, row in selected.iterrows():
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        axes[0].imshow(Image.open(row['ref_image_path']).convert('RGB'))
        axes[0].set_title(f"{row['base_id']} reference")
        axes[1].imshow(Image.open(row['generated_image_path']).convert('RGB'))
        axes[1].set_title(
            f"source {row['generated_source_id']} | {row['mapping_status']}"
        )
        for axis in axes:
            axis.axis('off')
        fig.suptitle(caption_text_from_row(row), fontsize=9, wrap=True)
        plt.tight_layout()
        plt.show()

show_mapping_audit(train_all_df)
display(mapping_manifest.query("mapping_status != 'identity'").head(25))
print('Saved mapping manifest:', OUTPUT_DIR / 'cc_mapping_manifest.tsv')


In [ ]:
def prompt_hash(text: str, length: int = 10) -> str:
    return hashlib.sha256(text.strip().encode('utf-8')).hexdigest()[:length]

def image_to_data_url(path: str) -> str:
    path = Path(path)
    media_type = {
        '.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
        '.webp': 'image/webp',
    }.get(path.suffix.lower())
    if media_type is None:
        raise ValueError(f'Unsupported image type: {path}')
    payload = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{media_type};base64,{payload}'

def extract_caption(row: pd.Series) -> str:
    for column in ['caption', 'text', 'prompt']:
        if column in row and pd.notna(row[column]):
            return str(row[column])
    ignored = set(ALL_GOLD_COLUMNS + [
        'id', 'image_id', 'version', 'split', 'base_id', 'caption_version',
        'caption_version_key', 'category', 'ref_image_path', 'generated_image_path',
    ])
    values = [
        f'{column}: {row[column]}' for column in row.index
        if column not in ignored and pd.notna(row[column])
    ]
    if not values:
        raise KeyError('No caption column found')
    return '\n'.join(values)

def parse_json_object(text: str) -> dict:
    text = str(text).strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end < start:
        raise ValueError('Response did not contain a JSON object')
    value = json.loads(text[start:end + 1])
    if not isinstance(value, dict):
        raise ValueError('Expected a JSON object')
    return value

def load_jsonl_records(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Malformed JSONL at {path}:{line_number}') from exc
    return records

def index_by_key(records, key: str, label: str) -> dict:
    indexed = {}
    for record in records:
        value = str(record[key])
        if value in indexed:
            raise ValueError(f'Duplicate {key}={value!r} in {label}')
        indexed[value] = record
    return indexed

def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()

def cache_coverage(frame: pd.DataFrame, records, key='instance_id') -> dict:
    requested = frame['id'].astype(str).tolist()
    available = {str(record[key]) for record in records}
    missing = [value for value in requested if value not in available]
    return {'expected': len(requested), 'available': len(requested) - len(missing), 'missing': missing}


In [ ]:
CC_PROMPT_VERSION = 'qatari-label-anchored-cc-v3'
CC_SCHEMA_VERSION = 'qatari-five-band-cc-schema-v3'
CC_DEMONSTRATION_RULE = (
    'six multimodal gold-labelled train examples; low-detail images; '
    'all demonstration base groups excluded from train judge evaluation'
)

SCENE_CATEGORIES = {
    'people_practice', 'architecture_landmark', 'objects_market'
}
QATAR_CONTEXT_STATUSES = {'matched', 'partial', 'missing', 'conflicting'}
SCORE_BAND_KEYS = tuple(SCORE_ANCHORS)

                                                                                
                                                                                  
CC_DEMONSTRATION_SPECS = [
    {
        'id': 'img_007_v3',
        'scene_category': 'people_practice',
        'qatar_context_status': 'conflicting',
        'brief_reason': (
            'The people and formal context do not preserve the intended Qatari scene.'
        ),
    },
    {
        'id': 'img_001_v5',
        'scene_category': 'people_practice',
        'qatar_context_status': 'partial',
        'brief_reason': (
            'The activity and coastal setting are partly coherent, but much of the '
            'intended Qatari context is lost.'
        ),
    },
    {
        'id': 'img_035_v3',
        'scene_category': 'objects_market',
        'qatar_context_status': 'partial',
        'brief_reason': (
            'The heritage-market organization is coherent but only partially preserves '
            'the intended Qatari context.'
        ),
    },
    {
        'id': 'img_007_v4',
        'scene_category': 'people_practice',
        'qatar_context_status': 'matched',
        'brief_reason': (
            'The portrait, attire, and formal setting form a coherent Qatari scene.'
        ),
    },
    {
        'id': 'img_029_v4',
        'scene_category': 'architecture_landmark',
        'qatar_context_status': 'conflicting',
        'brief_reason': (
            'The plausible coastal skyline does not preserve the intended Qatari '
            'landmark and spatial context.'
        ),
    },
    {
        'id': 'img_028_v4',
        'scene_category': 'architecture_landmark',
        'qatar_context_status': 'matched',
        'brief_reason': (
            'The Qatari architectural identity and its surrounding context are strongly '
            'preserved.'
        ),
    },
]
CC_DEMONSTRATION_IDS = [item['id'] for item in CC_DEMONSTRATION_SPECS]
CC_DEMONSTRATION_BASE_IDS = {
    parse_image_base_id(value) for value in CC_DEMONSTRATION_IDS
}

def score_to_band_probabilities(score: float) -> dict[str, float]:
    score = float(np.clip(score, 0.0, 1.0))
    keys = list(SCORE_ANCHORS)
    values = np.asarray([SCORE_ANCHORS[key] for key in keys], dtype=float)
    probabilities = np.zeros(len(values), dtype=float)
    if score <= values[0]:
        probabilities[0] = 1.0
    elif score >= values[-1]:
        probabilities[-1] = 1.0
    else:
        upper = int(np.searchsorted(values, score, side='right'))
        lower = upper - 1
        distance = values[upper] - values[lower]
        probabilities[upper] = (score - values[lower]) / distance
        probabilities[lower] = 1.0 - probabilities[upper]
    return {
        key: round(float(probability), 6)
        for key, probability in zip(keys, probabilities)
    }

def demonstration_row(instance_id: str) -> pd.Series:
    selected = train_all_df.loc[train_all_df['id'].eq(instance_id)]
    if len(selected) != 1:
        raise ValueError(f'Demonstration {instance_id!r} was not found uniquely')
    row = selected.iloc[0]
    if not bool(row['eligible_for_cc']):
        raise ValueError(f'Demonstration {instance_id!r} has unresolved mapping')
    return row

def demonstration_response(spec: dict) -> dict:
    row = demonstration_row(spec['id'])
    return {
        'scene_category': spec['scene_category'],
        'qatar_context_status': spec['qatar_context_status'],
        'score_probabilities': score_to_band_probabilities(row[TARGET_COLUMN]),
        'brief_reason': spec['brief_reason'],
    }

CC_DEMONSTRATION_RESPONSES = {
    spec['id']: demonstration_response(spec)
    for spec in CC_DEMONSTRATION_SPECS
}

                                                                                
train_judge_df = train_df.loc[
    ~train_df['base_id'].isin(CC_DEMONSTRATION_BASE_IDS)
].reset_index(drop=True)
assert not set(train_judge_df['base_id']) & CC_DEMONSTRATION_BASE_IDS

STRUCTURED_DIRECT_CC_PROMPT = r"""
ROLE

You are a multimodal judge of Contextual Coherence (CC) for a Qatar-focused
cultural image dataset. Return valid JSON only.

TARGET CULTURE

Every reference image concerns Qatari people, places, practices, objects, or
cultural settings. Qatar is the only target culture. Do not treat "Gulf" or "Arab"
as alternative correct target cultures. A broader regional scene is a mismatch; it
may receive partial CC only when the complete candidate remains contextually coherent
and the current caption is broad. It cannot receive full Qatar-context credit.

INPUTS

You receive:
1. A Qatari reference image.
2. Its culturally explicit V1 source caption.
3. The current caption used to generate the candidate.
4. The generated candidate image.

TASK

Judge whether the visible elements in the candidate are placed together in a context
appropriate to the intended scene.

CC is not only caption-image similarity, not only correctness of isolated cultural
elements, and not only the quantity of Qatar-specific cues. Use the current caption
to determine requested content and relations. Use the reference image and V1 caption
to identify the intended Qatari scene, place, practice, and contextual organization.
Do not demand exact details that the current caption clearly omits unless their loss
makes the setting contextually wrong or generic.

CATEGORY-SPECIFIC RULES

Architecture or landmarks:
- Strongly weight the identity of the actual Qatari landmark, its surroundings, and
  spatial context.
- A coherent foreign, broader-regional, or generic landmark is not an appropriate
  replacement.
- Usually decide among strong match, partial resemblance, and clear mismatch.

People or cultural practices:
- Judge whether people, attire, activity, objects, and environment belong together.
- Correct Qatari attire alone does not guarantee high CC if the wider setting or
  activity is absent or inappropriate.
- A coherent but non-Qatari regional scene can receive partial, never full, credit
  when the current caption is broad.

Objects, crafts, or markets:
- Judge whether objects are placed, displayed, and used in an appropriate Qatari
  heritage, domestic, commercial, or activity setting.
- A generic bazaar or heritage setting receives only partial credit when its
  organization is plausible but Qatar-specific context is lost.

METRIC SEPARATION

- CEA evaluates correctness of individual cultural elements.
- CS evaluates cultural distinctiveness and Qatar-specificity.
- CC evaluates whether the complete scene and visible relations make contextual sense.
- A wrong national identity reduces CC and prevents a full score, but does not force
  zero when a broad current caption and the complete scene remain coherent.
- Ignore photorealism and aesthetics unless an artifact prevents interpretation.

SCORE BANDS

0.00: The intended setting is absent, generic when context is essential, visibly
wrong, or contradicted; important elements do not belong together.

0.25: Only weak contextual evidence is preserved; major setting, landmark, activity,
or cultural relations are incorrect.

0.50: The scene is meaningfully but incompletely coherent; it preserves part of the
intended context or presents a related setting while losing important Qatari context.

0.75: The scene is mostly contextually appropriate; primary setting and relations are
coherent, with a limited omission, substitution, or national-specificity mismatch.

1.00: The candidate strongly preserves the intended Qatari scene or setting, and all
important people, objects, activities, and spatial relations are appropriate.

OUTPUT

Return probabilities over the five score bands. Do not force certainty. All five
probabilities must be present, lie in [0,1], and sum approximately to 1.

{
  "scene_category": "people_practice | architecture_landmark | objects_market",
  "qatar_context_status": "matched | partial | missing | conflicting",
  "score_probabilities": {
    "0.00": 0.0,
    "0.25": 0.0,
    "0.50": 0.0,
    "0.75": 0.0,
    "1.00": 0.0
  },
  "brief_reason": "One sentence grounded in visible contextual evidence."
}

Return no markdown and no text outside the JSON object.
"""

CC_REQUIRED_TOP_LEVEL = {
    'scene_category', 'qatar_context_status',
    'score_probabilities', 'brief_reason'
}
CC_SCHEMA_CANONICAL = {
    'required': sorted(CC_REQUIRED_TOP_LEVEL),
    'scene_categories': sorted(SCENE_CATEGORIES),
    'qatar_context_statuses': sorted(QATAR_CONTEXT_STATUSES),
    'score_bands': SCORE_ANCHORS,
}

def cc_demonstration_signature() -> str:
    payload = []
    for spec in CC_DEMONSTRATION_SPECS:
        row = demonstration_row(spec['id'])
        payload.append({
            'id': spec['id'],
            'generated_source_id': str(row['generated_source_id']),
            'reference_sha256': file_sha256(row['ref_image_path']),
            'generated_sha256': file_sha256(row['generated_image_path']),
            'response': CC_DEMONSTRATION_RESPONSES[spec['id']],
        })
    return prompt_hash(json.dumps(payload, sort_keys=True), length=16)

def cc_cache_tag() -> str:
    mapping_signature = prompt_hash(json.dumps({
        'version': MAPPING_AUDIT_VERSION,
        'repair_enabled': ENABLE_VERIFIED_TRAIN_MAPPING_REPAIR,
        'remap': VERIFIED_GENERATED_BASE_REMAP,
        'excluded': {
            key: sorted(value) for key, value in UNRESOLVED_BASE_IDS.items()
        },
    }, sort_keys=True))
    return (
        f'{CC_PROMPT_VERSION}_{CC_MODEL}_reasoning-{CC_REASONING_EFFORT}_'
        f'detail-{CC_IMAGE_DETAIL}_prompt-{prompt_hash(STRUCTURED_DIRECT_CC_PROMPT)}_'
        f'schema-{prompt_hash(json.dumps(CC_SCHEMA_CANONICAL, sort_keys=True))}_'
        f'mapping-{mapping_signature}_demos-{cc_demonstration_signature()}'
    )

def cc_cache_path(split: str) -> Path:
    return CACHE_DIR / f'structured_cc_{split}_{cc_cache_tag()}.jsonl'

def cc_attempt_cache_path(split: str) -> Path:
    return CACHE_DIR / f'structured_cc_attempts_{split}_{cc_cache_tag()}.jsonl'

print('CC configuration:', cc_cache_tag())
print('Demonstrations:', CC_DEMONSTRATION_IDS)
print('Demonstration base groups excluded from train evaluation:',
      sorted(CC_DEMONSTRATION_BASE_IDS))
print('Train judge/evaluation rows:', len(train_judge_df), '| groups:',
      train_judge_df['base_id'].nunique())
display(pd.DataFrame([
    {
        'id': spec['id'],
        'gold_cc': float(demonstration_row(spec['id'])[TARGET_COLUMN]),
        **CC_DEMONSTRATION_RESPONSES[spec['id']],
    }
    for spec in CC_DEMONSTRATION_SPECS
]))


In [ ]:
def finite_unit_interval(value, name: str) -> float:
    number = float(value)
    if not np.isfinite(number) or not 0.0 <= number <= 1.0:
        raise ValueError(f'{name} must be finite and in [0, 1]; got {value!r}')
    return number

def validate_structured_cc_response(value: dict, instance_id: str) -> dict:
    missing = CC_REQUIRED_TOP_LEVEL - set(value)
    if missing:
        raise ValueError(f'Missing top-level fields: {sorted(missing)}')

    scene_category = str(value['scene_category'])
    qatar_status = str(value['qatar_context_status'])
    if scene_category not in SCENE_CATEGORIES:
        raise ValueError(f'Invalid scene_category: {scene_category!r}')
    if qatar_status not in QATAR_CONTEXT_STATUSES:
        raise ValueError(f'Invalid qatar_context_status: {qatar_status!r}')

    probabilities = value['score_probabilities']
    if not isinstance(probabilities, dict):
        raise ValueError('score_probabilities must be an object')
    if set(probabilities) != set(SCORE_BAND_KEYS):
        raise ValueError(
            'score_probabilities must contain exactly '
            f'{list(SCORE_BAND_KEYS)}; got {sorted(probabilities)}'
        )
    canonical_probabilities = {
        key: finite_unit_interval(probabilities[key], f'probability_{key}')
        for key in SCORE_BAND_KEYS
    }
    probability_sum = float(sum(canonical_probabilities.values()))
    if not 0.98 <= probability_sum <= 1.02:
        raise ValueError(
            f'score_probabilities must sum approximately to 1; got {probability_sum}'
        )
    canonical_probabilities = {
        key: value / probability_sum
        for key, value in canonical_probabilities.items()
    }
    raw_cc = float(sum(
        SCORE_ANCHORS[key] * canonical_probabilities[key]
        for key in SCORE_BAND_KEYS
    ))
    reason = str(value['brief_reason']).strip()
    if not reason:
        raise ValueError('brief_reason is empty')
    return {
        'scene_category': scene_category,
        'qatar_context_status': qatar_status,
        'score_probabilities': canonical_probabilities,
        'raw_cc': raw_cc,
        'brief_reason': reason,
    }

def structured_cc_user_content(
    row: pd.Series, image_detail: str, labelled_example: bool = False
) -> list[dict]:
    prefix = (
        'LABELLED TRAINING EXAMPLE' if labelled_example
        else 'UNLABELLED INSTANCE TO JUDGE'
    )
    text = f"""{prefix}
Instance ID: {row['id']}

V1 caption (Qatari source context):
{get_v1_caption(row)}

Current caption (v{row['caption_version']}):
{extract_caption(row)}

The first image is the Qatari reference. The second is the generated candidate.
Apply the category-specific CC rubric and return JSON only."""
    return [
        {'type': 'input_text', 'text': text},
        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
        {
            'type': 'input_image',
            'image_url': image_to_data_url(row['ref_image_path']),
            'detail': image_detail,
        },
        {'type': 'input_text', 'text': 'GENERATED IMAGE:'},
        {
            'type': 'input_image',
            'image_url': image_to_data_url(row['generated_image_path']),
            'detail': image_detail,
        },
    ]

def cc_demonstration_messages() -> list[dict]:
    messages = []
    for spec in CC_DEMONSTRATION_SPECS:
        row = demonstration_row(spec['id'])
        messages.append({
            'role': 'user',
            'content': structured_cc_user_content(
                row,
                image_detail=CC_DEMONSTRATION_IMAGE_DETAIL,
                labelled_example=True,
            ),
        })
        messages.append({
            'role': 'assistant',
            'content': json.dumps(
                CC_DEMONSTRATION_RESPONSES[spec['id']],
                ensure_ascii=False,
            ),
        })
    return messages

def call_structured_cc(row: pd.Series, split: str) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        timestamp = datetime.now(timezone.utc).isoformat()
        try:
            response = get_openai_client().responses.create(
                model=CC_MODEL,
                reasoning={'effort': CC_REASONING_EFFORT},
                input=[
                    {
                        'role': 'developer',
                        'content': STRUCTURED_DIRECT_CC_PROMPT.strip(),
                    },
                    *cc_demonstration_messages(),
                    {
                        'role': 'user',
                        'content': structured_cc_user_content(
                            row, image_detail=CC_IMAGE_DETAIL
                        ),
                    },
                ],
            )
            parsed = parse_json_object(response.output_text)
            canonical = validate_structured_cc_response(parsed, str(row['id']))
            record = {
                'instance_id': str(row['id']),
                'generated_source_id': str(row['generated_source_id']),
                'mapping_audit_version': MAPPING_AUDIT_VERSION,
                'prompt_version': CC_PROMPT_VERSION,
                'prompt_hash': prompt_hash(STRUCTURED_DIRECT_CC_PROMPT),
                'schema_version': CC_SCHEMA_VERSION,
                'schema_hash': prompt_hash(
                    json.dumps(CC_SCHEMA_CANONICAL, sort_keys=True)
                ),
                'model': CC_MODEL,
                'reasoning_effort': CC_REASONING_EFFORT,
                'image_detail': CC_IMAGE_DETAIL,
                'demonstration_image_detail': CC_DEMONSTRATION_IMAGE_DETAIL,
                'demonstration_ids': CC_DEMONSTRATION_IDS,
                'demonstration_signature': cc_demonstration_signature(),
                'demonstration_rule': CC_DEMONSTRATION_RULE,
                'parsing_status': 'valid',
                'created_utc': timestamp,
                'response': canonical,
            }
            append_jsonl(cc_attempt_cache_path(split), {
                **{key: record[key] for key in record if key != 'response'},
                'attempt': attempt,
                'raw_response': response.output_text,
            })
            return record
        except Exception as exc:
            append_jsonl(cc_attempt_cache_path(split), {
                'instance_id': str(row['id']),
                'generated_source_id': str(row['generated_source_id']),
                'attempt': attempt,
                'mapping_audit_version': MAPPING_AUDIT_VERSION,
                'prompt_version': CC_PROMPT_VERSION,
                'prompt_hash': prompt_hash(STRUCTURED_DIRECT_CC_PROMPT),
                'schema_version': CC_SCHEMA_VERSION,
                'model': CC_MODEL,
                'parsing_status': 'invalid',
                'error': repr(exc),
                'created_utc': timestamp,
            })
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def validate_cc_cache_record(record: dict) -> dict:
    expected = {
        'mapping_audit_version': MAPPING_AUDIT_VERSION,
        'prompt_version': CC_PROMPT_VERSION,
        'prompt_hash': prompt_hash(STRUCTURED_DIRECT_CC_PROMPT),
        'schema_version': CC_SCHEMA_VERSION,
        'schema_hash': prompt_hash(json.dumps(CC_SCHEMA_CANONICAL, sort_keys=True)),
        'model': CC_MODEL,
        'reasoning_effort': CC_REASONING_EFFORT,
        'image_detail': CC_IMAGE_DETAIL,
        'demonstration_image_detail': CC_DEMONSTRATION_IMAGE_DETAIL,
        'demonstration_ids': CC_DEMONSTRATION_IDS,
        'demonstration_signature': cc_demonstration_signature(),
        'demonstration_rule': CC_DEMONSTRATION_RULE,
        'parsing_status': 'valid',
    }
    for key, expected_value in expected.items():
        if record.get(key) != expected_value:
            raise ValueError(
                f'Incompatible CC cache metadata for '
                f'{record.get("instance_id")}: {key}'
            )
    output = dict(record)
    output['response'] = validate_structured_cc_response(
        output['response'], str(output['instance_id'])
    )
    return output

def load_or_infer_structured_cc(
    frame: pd.DataFrame, split: str, allow_calls: bool
) -> pd.DataFrame:
    path = cc_cache_path(split)
    cached = index_by_key(load_jsonl_records(path), 'instance_id', str(path))
    for key in list(cached):
        cached[key] = validate_cc_cache_record(cached[key])
    coverage = cache_coverage(frame, cached.values())
    print(f'CC {split} cache: {coverage["available"]}/{coverage["expected"]}')
    if coverage['missing']:
        print(
            'Missing CC IDs:', coverage['missing'][:20],
            '...' if len(coverage['missing']) > 20 else ''
        )
    if allow_calls:
        rows = frame.head(RUN_LIMIT) if RUN_LIMIT is not None else frame
        with tqdm(total=len(rows), desc=f'Qatari label-anchored CC {split}') as progress:
            for _, row in rows.iterrows():
                instance_id = str(row['id'])
                if instance_id not in cached:
                    record = call_structured_cc(row, split)
                    append_jsonl(path, record)
                    cached[instance_id] = record
                    time.sleep(REQUEST_SLEEP_SECONDS)
                progress.update(1)
    ordered = [
        cached[str(value)] for value in frame['id'] if str(value) in cached
    ]
    return pd.DataFrame(ordered)

train_cc_records = load_or_infer_structured_cc(
    train_judge_df,
    'train',
    allow_calls=(RUN_NEW_API_CALLS and not PILOT_DEV_ONLY),
)
dev_cc_records = load_or_infer_structured_cc(
    dev_df,
    'dev',
    allow_calls=RUN_NEW_API_CALLS,
)
test_cc_records = (
    load_or_infer_structured_cc(
        test_df, 'test', allow_calls=RUN_TEST_API_CALLS
    )
    if test_df is not None else pd.DataFrame()
)


In [ ]:
PROBABILITY_COLUMNS = {
    key: f'probability_{key.replace(".", "")}' for key in SCORE_BAND_KEYS
}

def cc_records_to_features(
    records: pd.DataFrame, metadata: pd.DataFrame
) -> pd.DataFrame:
    empty_columns = [
        'id', 'base_id', 'caption_version', 'caption_version_key', 'category',
        'raw_prediction', 'judge_confidence', 'score_entropy',
        'scene_category', 'qatar_context_status', *PROBABILITY_COLUMNS.values(),
    ]
    if records.empty:
        return pd.DataFrame(columns=empty_columns)
    output = []
    for record in records.to_dict('records'):
        response = record['response']
        probabilities = response['score_probabilities']
        probability_values = np.asarray([
            probabilities[key] for key in SCORE_BAND_KEYS
        ], dtype=float)
        positive = probability_values[probability_values > 0]
        entropy = -float(np.sum(positive * np.log(positive)))
        row = {
            'id': str(record['instance_id']),
            'raw_prediction': float(response['raw_cc']),
            'judge_confidence': float(probability_values.max()),
            'score_entropy': entropy,
            'scene_category': str(response['scene_category']),
            'qatar_context_status': str(response['qatar_context_status']),
        }
        row.update({
            PROBABILITY_COLUMNS[key]: float(probabilities[key])
            for key in SCORE_BAND_KEYS
        })
        output.append(row)
    result = pd.DataFrame(output)
    metadata_columns = [
        'id', 'base_id', 'caption_version', 'caption_version_key', 'category'
    ]
    result = metadata[metadata_columns].merge(
        result, on='id', how='inner', validate='one_to_one'
    )
    assert result['raw_prediction'].between(0, 1).all()
    assert not result[['raw_prediction', *PROBABILITY_COLUMNS.values()]].isna().any().any()
    return result

train_cc_features = cc_records_to_features(train_cc_records, train_judge_df)
dev_cc_features = cc_records_to_features(dev_cc_records, dev_df)
test_cc_features = (
    cc_records_to_features(test_cc_records, test_df)
    if test_df is not None else pd.DataFrame()
)
print('CC features:', len(train_cc_features), 'train |',
      len(dev_cc_features), 'dev |', len(test_cc_features), 'test')
display(dev_cc_features.head())


In [ ]:
def safe_spearman(gold, predicted) -> float:
    gold = pd.Series(gold, dtype=float)
    predicted = pd.Series(predicted, dtype=float)
    if len(gold) < 2 or gold.nunique() < 2 or predicted.nunique() < 2:
        return np.nan
    return float(spearmanr(gold, predicted).correlation)

def assert_predictions(values, label: str):
    values = np.asarray(values, dtype=float)
    assert np.isfinite(values).all(), f'{label}: predictions contain NaN/inf'
    assert ((0 <= values) & (values <= 1)).all(), f'{label}: outside [0, 1]'

def fold_splits(frame: pd.DataFrame, n_splits: int):
    groups = frame['base_id'].astype(str).to_numpy()
    n_splits = min(n_splits, pd.Series(groups).nunique())
    if n_splits < 2:
        raise ValueError('At least two reference-image groups are required')
    splitter = GroupKFold(n_splits=n_splits)
    for train_index, validation_index in splitter.split(frame, groups=groups):
        assert set(groups[train_index]).isdisjoint(set(groups[validation_index]))
        yield train_index, validation_index

def fit_version_prior(training: pd.DataFrame):
    medians = training.groupby('caption_version')[GOLD_COLUMN].median()
    fallback = float(training[GOLD_COLUMN].median())
    return medians, fallback

def predict_version_prior(training: pd.DataFrame, target: pd.DataFrame):
    medians, fallback = fit_version_prior(training)
    return target['caption_version'].map(medians).fillna(fallback).to_numpy(float)

def choose_blend_weight(frame: pd.DataFrame, n_splits: int):
    rows = []
    for weight in BLEND_WEIGHTS:
        correlations, maes = [], []
        for train_index, validation_index in fold_splits(frame, n_splits):
            training = frame.iloc[train_index]
            validation = frame.iloc[validation_index]
            prior = predict_version_prior(training, validation)
            raw = validation['raw_prediction'].to_numpy(float)
            prediction = np.clip(weight * raw + (1.0 - weight) * prior, 0, 1)
            correlations.append(safe_spearman(validation[GOLD_COLUMN], prediction))
            maes.append(mean_absolute_error(validation[GOLD_COLUMN], prediction))
        rows.append({
            'raw_weight': float(weight),
            'mean_spearman': float(np.nanmean(correlations)),
            'mean_mae': float(np.mean(maes)),
        })
    table = pd.DataFrame(rows)
    best = table.sort_values(
        ['mean_spearman', 'mean_mae', 'raw_weight'],
        ascending=[False, True, False],
    ).iloc[0]
    return float(best['raw_weight']), table

def summarize_fold_metrics(fold_metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for variant, group in fold_metrics.groupby('variant'):
        values = group['spearman'].dropna().astype(float)
        rows.append({
            'variant': variant,
            'cv_spearman_mean': values.mean() if len(values) else np.nan,
            'cv_spearman_std': values.std(ddof=1) if len(values) > 1 else np.nan,
            'cv_spearman_min': values.min() if len(values) else np.nan,
            'cv_spearman_max': values.max() if len(values) else np.nan,
            'cv_mae_mean': group['mae'].mean(),
            'positive_vs_prior_folds': int(group['delta_vs_prior'].gt(0).sum()),
            'folds': len(group),
        })
    return pd.DataFrame(rows)

def fit_nested_blend(frame: pd.DataFrame) -> dict:
    required = {
        'id', 'base_id', 'caption_version', GOLD_COLUMN, 'raw_prediction'
    }
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'Blend input missing columns: {sorted(missing)}')
    variants = ['version_prior', 'raw', 'nested_blend']
    oof = {name: np.full(len(frame), np.nan) for name in variants}
    fold_rows = []
    weight_rows = []
    for fold, (train_index, validation_index) in enumerate(
        fold_splits(frame, OUTER_FOLDS), start=1
    ):
        training = frame.iloc[train_index]
        validation = frame.iloc[validation_index]
        weight, weight_table = choose_blend_weight(training, INNER_FOLDS)
        prior = predict_version_prior(training, validation)
        raw = validation['raw_prediction'].to_numpy(float)
        predictions = {
            'version_prior': prior,
            'raw': raw,
            'nested_blend': np.clip(weight * raw + (1.0 - weight) * prior, 0, 1),
        }
        prior_correlation = safe_spearman(validation[GOLD_COLUMN], prior)
        for variant, prediction in predictions.items():
            assert_predictions(prediction, f'{variant}/fold{fold}')
            oof[variant][validation_index] = prediction
            correlation = safe_spearman(validation[GOLD_COLUMN], prediction)
            fold_rows.append({
                'fold': fold,
                'variant': variant,
                'spearman': correlation,
                'mae': mean_absolute_error(validation[GOLD_COLUMN], prediction),
                'delta_vs_prior': correlation - prior_correlation,
                'selected_raw_weight': weight,
                'n_rows': len(validation_index),
                'n_groups': validation['base_id'].nunique(),
                'validation_groups': ','.join(
                    sorted(validation['base_id'].astype(str).unique())
                ),
            })
        weight_rows.append(weight_table.assign(outer_fold=fold))
    for name, values in oof.items():
        assert_predictions(values, f'{name}/OOF')
    fold_metrics = pd.DataFrame(fold_rows)
    summary = summarize_fold_metrics(fold_metrics)
    final_weight, final_weight_table = choose_blend_weight(frame, OUTER_FOLDS)

    prior_mean = float(summary.loc[
        summary['variant'].eq('version_prior'), 'cv_spearman_mean'
    ].iloc[0])
    eligible = []
    for variant in ['raw', 'nested_blend']:
        row = summary.loc[summary['variant'].eq(variant)].iloc[0]
        deltas = fold_metrics.loc[
            fold_metrics['variant'].eq(variant), 'delta_vs_prior'
        ].dropna()
        stable = (
            row['cv_spearman_mean'] >= (
                prior_mean + MIN_BLEND_GAIN_OVER_VERSION_PRIOR
            )
            and deltas.gt(0).sum() >= int(np.ceil(len(deltas) / 2))
            and deltas.min() > -0.10
        )
        if stable:
            eligible.append((float(row['cv_spearman_mean']), variant))
    selected_variant = (
        sorted(eligible, reverse=True)[0][1] if eligible else 'version_prior'
    )
    return {
        'frame': frame.copy(),
        'oof': oof,
        'fold_metrics': fold_metrics,
        'summary': summary,
        'outer_weight_tables': pd.concat(weight_rows, ignore_index=True),
        'final_weight': final_weight,
        'final_weight_table': final_weight_table,
        'selected_variant': selected_variant,
    }

def predict_blend(result: dict, prior_training: pd.DataFrame,
                  target_features: pd.DataFrame) -> pd.DataFrame:
    output = target_features[[
        'id', 'base_id', 'caption_version', 'caption_version_key', 'category'
    ]].copy()
    output['version_prior'] = predict_version_prior(
        prior_training, target_features
    )
    output['raw'] = target_features['raw_prediction'].to_numpy(float)
    output['nested_blend'] = np.clip(
        result['final_weight'] * output['raw']
        + (1.0 - result['final_weight']) * output['version_prior'],
        0, 1,
    )
    output['selected_prediction'] = output[result['selected_variant']]
    for column in ['version_prior', 'raw', 'nested_blend', 'selected_prediction']:
        assert_predictions(output[column], f'final/{column}')
    return output


In [ ]:
def gold_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[['id', 'base_id', 'caption_version', TARGET_COLUMN]].rename(
        columns={TARGET_COLUMN: GOLD_COLUMN}
    ).copy()

def caption_median_baseline(train_metadata, target_metadata):
    training = gold_frame(train_metadata)
    oof = np.full(len(training), np.nan)
    fold_rows = []
    for fold, (train_index, validation_index) in enumerate(
        fold_splits(training, OUTER_FOLDS), start=1
    ):
        train_fold = training.iloc[train_index]
        validation = training.iloc[validation_index]
        prediction = predict_version_prior(train_fold, validation)
        oof[validation_index] = prediction
        fold_rows.append({
            'fold': fold,
            'variant': 'version_prior',
            'spearman': safe_spearman(validation[GOLD_COLUMN], prediction),
            'mae': mean_absolute_error(validation[GOLD_COLUMN], prediction),
            'n_rows': len(validation_index),
            'n_groups': validation['base_id'].nunique(),
        })
    target_prediction = target_metadata[[
        'id', 'base_id', 'caption_version'
    ]].copy()
    target_prediction['prediction'] = predict_version_prior(
        training, target_metadata
    )
    assert_predictions(oof, 'caption-version median/OOF')
    assert_predictions(
        target_prediction['prediction'], 'caption-version median/final'
    )
    return {
        'oof': oof,
        'fold_metrics': pd.DataFrame(fold_rows),
        'target_prediction': target_prediction,
    }

median_dev_result = caption_median_baseline(train_df, dev_df)
median_test_result = (
    caption_median_baseline(train_df, test_df) if test_df is not None else None
)

blend_result = None
dev_prediction = None
test_prediction = None
RECOMMENDED_SYSTEM = None

dev_complete = len(dev_cc_features) == len(dev_df)
train_complete = len(train_cc_features) == len(train_judge_df)

if dev_complete:
    prior_training = gold_frame(train_df)
    dev_prediction = dev_cc_features[[
        'id', 'base_id', 'caption_version', 'caption_version_key',
        'category', 'raw_prediction'
    ]].copy()
    dev_prediction['version_prior'] = predict_version_prior(
        prior_training, dev_prediction
    )

if train_complete:
    train_blend_frame = train_cc_features.merge(
        train_judge_df[['id', TARGET_COLUMN]],
        on='id', how='inner', validate='one_to_one'
    ).rename(columns={TARGET_COLUMN: GOLD_COLUMN})
    blend_result = fit_nested_blend(train_blend_frame)
    RECOMMENDED_SYSTEM = blend_result['selected_variant']
    print('Train-CV-selected variant:', RECOMMENDED_SYSTEM)
    print('Final raw weight:', blend_result['final_weight'])
    display(blend_result['summary'].round(4))
    display(blend_result['final_weight_table'].round(4))

    if dev_complete:
        dev_prediction = predict_blend(
            blend_result, gold_frame(train_df), dev_cc_features
        )
    if test_df is not None and len(test_cc_features) == len(test_df):
        test_prediction = predict_blend(
            blend_result, gold_frame(train_df), test_cc_features
        )
else:
    print(
        'Full train-CV selection pending train cache:',
        f'{len(train_cc_features)}/{len(train_judge_df)}'
    )

if not dev_complete:
    print('Dev pilot pending:', f'{len(dev_cc_features)}/{len(dev_df)}')


In [ ]:
def evaluate_final(gold_metadata, prediction_frame, prediction_column, label):
    merged = gold_metadata[[
        'id', 'base_id', 'caption_version', TARGET_COLUMN
    ]].merge(
        prediction_frame[['id', prediction_column]],
        on='id', how='inner', validate='one_to_one'
    )
    prediction = merged[prediction_column].astype(float)
    assert_predictions(prediction, label)
    return {
        'system': label,
        'dev_spearman': safe_spearman(merged[TARGET_COLUMN], prediction),
        'dev_mae': mean_absolute_error(merged[TARGET_COLUMN], prediction),
        'dev_rows': len(merged),
        'dev_groups': merged['base_id'].nunique(),
    }, merged.rename(columns={
        TARGET_COLUMN: 'gold', prediction_column: 'predicted'
    })

def by_caption_version(evaluation_frame, system):
    rows = []
    for version, group in evaluation_frame.groupby('caption_version'):
        rows.append({
            'system': system,
            'caption_version': f'v{int(version)}',
            'spearman': safe_spearman(group['gold'], group['predicted']),
            'mae': mean_absolute_error(group['gold'], group['predicted']),
            'n': len(group),
        })
    return pd.DataFrame(rows)

def cluster_bootstrap_spearman(evaluation_frame, replicates=BOOTSTRAP_REPLICATES):
    groups = evaluation_frame['base_id'].astype(str).unique()
    if len(groups) < 4:
        return (np.nan, np.nan)
    rng = np.random.default_rng(RANDOM_SEED)
    indexed = {
        group: evaluation_frame[
            evaluation_frame['base_id'].astype(str).eq(group)
        ] for group in groups
    }
    values = []
    for _ in range(replicates):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        draw = pd.concat([indexed[group] for group in sampled], ignore_index=True)
        value = safe_spearman(draw['gold'], draw['predicted'])
        if not np.isnan(value):
            values.append(value)
    return tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)

comparison_rows = []
evaluation_frames = {}
caption_version_reports = []

median_metric, median_evaluation = evaluate_final(
    dev_df,
    median_dev_result['target_prediction'],
    'prediction',
    'Caption-version median',
)
median_fold_metrics = median_dev_result['fold_metrics']
comparison_rows.append({
    **median_metric,
    'cv_spearman_mean': median_fold_metrics['spearman'].mean(),
    'cv_spearman_std': median_fold_metrics['spearman'].std(ddof=1),
    'notes': 'Non-visual baseline; full eligible train labels',
})
evaluation_frames['Caption-version median'] = median_evaluation
caption_version_reports.append(by_caption_version(
    median_evaluation, 'Caption-version median'
))

if dev_prediction is not None:
    raw_metric, raw_evaluation = evaluate_final(
        dev_df, dev_prediction, 'raw_prediction' if blend_result is None else 'raw',
        'Raw Qatari five-band judge'
    )
    raw_cv = (
        blend_result['summary'].loc[
            blend_result['summary']['variant'].eq('raw')
        ].iloc[0]
        if blend_result is not None else None
    )
    comparison_rows.append({
        **raw_metric,
        'cv_spearman_mean': (
            raw_cv['cv_spearman_mean'] if raw_cv is not None else np.nan
        ),
        'cv_spearman_std': (
            raw_cv['cv_spearman_std'] if raw_cv is not None else np.nan
        ),
        'notes': (
            f'Dev pilot target: > {RAW_PILOT_TARGET:.3f}; '
            'five-band expected value, no calibration'
        ),
    })
    evaluation_frames['Raw Qatari five-band judge'] = raw_evaluation
    caption_version_reports.append(by_caption_version(
        raw_evaluation, 'Raw Qatari five-band judge'
    ))

if blend_result is not None and dev_prediction is not None:
    blend_metric, blend_evaluation = evaluate_final(
        dev_df, dev_prediction, 'nested_blend', 'Nested-CV one-parameter blend'
    )
    blend_cv = blend_result['summary'].loc[
        blend_result['summary']['variant'].eq('nested_blend')
    ].iloc[0]
    comparison_rows.append({
        **blend_metric,
        'cv_spearman_mean': blend_cv['cv_spearman_mean'],
        'cv_spearman_std': blend_cv['cv_spearman_std'],
        'notes': f'Final raw weight={blend_result["final_weight"]:.2f}',
    })
    evaluation_frames['Nested-CV one-parameter blend'] = blend_evaluation
    caption_version_reports.append(by_caption_version(
        blend_evaluation, 'Nested-CV one-parameter blend'
    ))

comparison_table = pd.DataFrame(comparison_rows)
for index, row in comparison_table.iterrows():
    evaluation = evaluation_frames[row['system']]
    low, high = cluster_bootstrap_spearman(evaluation)
    comparison_table.loc[index, 'dev_spearman_cluster_ci95'] = (
        f'[{low:.3f}, {high:.3f}]'
    )

caption_version_table = pd.concat(
    caption_version_reports, ignore_index=True
) if caption_version_reports else pd.DataFrame()

PILOT_DECISION = 'pending'
if 'Raw Qatari five-band judge' in evaluation_frames:
    raw_dev = float(comparison_table.loc[
        comparison_table['system'].eq('Raw Qatari five-band judge'),
        'dev_spearman'
    ].iloc[0])
    PILOT_DECISION = (
        'promising: proceed to train calls'
        if raw_dev > RAW_PILOT_TARGET
        else 'not promising: stop; do not run train or test calls'
    )
    print('Pilot decision:', PILOT_DECISION)
    print('Raw dev Spearman:', round(raw_dev, 4),
          '| required to proceed:', f'> {RAW_PILOT_TARGET:.3f}')

print('Frozen train-CV recommendation:', RECOMMENDED_SYSTEM or 'pending')
display(comparison_table.round(4))
display(caption_version_table.round(4))
if blend_result is not None:
    display(blend_result['fold_metrics'].round(4))

comparison_table.to_csv(
    OUTPUT_DIR / 'cc_v3_system_comparison.tsv', sep='\t', index=False
)
caption_version_table.to_csv(
    OUTPUT_DIR / 'cc_v3_by_caption_version.tsv', sep='\t', index=False
)
if blend_result is not None:
    blend_result['fold_metrics'].to_csv(
        OUTPUT_DIR / 'cc_v3_grouped_cv_fold_metrics.tsv',
        sep='\t', index=False
    )
    blend_result['final_weight_table'].to_csv(
        OUTPUT_DIR / 'cc_v3_blend_weight_search.tsv',
        sep='\t', index=False
    )


In [ ]:
train_export = train_judge_df[[
    'id', 'base_id', 'caption_version', 'generated_source_id', TARGET_COLUMN
]].rename(columns={TARGET_COLUMN: GOLD_COLUMN}).copy()

if blend_result is not None:
    for variant, values in blend_result['oof'].items():
        train_export[f'{variant}_oof'] = values
    train_export['selected_prediction_oof'] = blend_result['oof'][
        blend_result['selected_variant']
    ]
    assert_predictions(
        train_export['selected_prediction_oof'], 'train selected OOF'
    )

dev_export = dev_df[[
    'id', 'base_id', 'caption_version', 'generated_source_id'
]].copy()
dev_export = dev_export.merge(
    median_dev_result['target_prediction'][['id', 'prediction']].rename(
        columns={'prediction': 'caption_version_median'}
    ), on='id', validate='one_to_one'
)

if dev_prediction is not None:
    raw_column = 'raw' if 'raw' in dev_prediction.columns else 'raw_prediction'
    dev_export = dev_export.merge(
        dev_prediction[['id', raw_column]].rename(
            columns={raw_column: 'raw_prediction'}
        ), on='id', validate='one_to_one'
    )
    if 'nested_blend' in dev_prediction.columns:
        dev_export = dev_export.merge(
            dev_prediction[['id', 'nested_blend']],
            on='id', validate='one_to_one'
        )
    if RECOMMENDED_SYSTEM is not None:
        source = {
            'version_prior': 'caption_version_median',
            'raw': 'raw_prediction',
            'nested_blend': 'nested_blend',
        }[RECOMMENDED_SYSTEM]
        dev_export['final_prediction'] = dev_export[source]
        dev_export['CRAI_CC'] = dev_export['final_prediction']
        assert_predictions(dev_export['final_prediction'], 'dev final prediction')

test_export = None
if test_df is not None and RECOMMENDED_SYSTEM == 'version_prior':
                                                          
    test_export = test_df[[
        'id', 'base_id', 'caption_version', 'generated_source_id'
    ]].copy()
    test_export = test_export.merge(
        median_test_result['target_prediction'][['id', 'prediction']].rename(
            columns={'prediction': 'caption_version_median'}
        ), on='id', validate='one_to_one'
    )
    test_export['final_prediction'] = test_export['caption_version_median']
    test_export['CRAI_CC'] = test_export['final_prediction']
    assert_predictions(test_export['final_prediction'], 'test final prediction')
elif test_prediction is not None and RECOMMENDED_SYSTEM is not None:
    test_export = test_df[[
        'id', 'base_id', 'caption_version', 'generated_source_id'
    ]].copy()
    test_export = test_export.merge(
        median_test_result['target_prediction'][['id', 'prediction']].rename(
            columns={'prediction': 'caption_version_median'}
        ), on='id', validate='one_to_one'
    )
    test_export = test_export.merge(
        test_prediction[['id', 'raw', 'nested_blend']].rename(
            columns={'raw': 'raw_prediction'}
        ), on='id', validate='one_to_one'
    )
    source = {
        'version_prior': 'caption_version_median',
        'raw': 'raw_prediction',
        'nested_blend': 'nested_blend',
    }[RECOMMENDED_SYSTEM]
    test_export['final_prediction'] = test_export[source]
    test_export['CRAI_CC'] = test_export['final_prediction']
    assert_predictions(test_export['final_prediction'], 'test final prediction')

train_export.to_csv(
    OUTPUT_DIR / 'cc_v3_train_oof_predictions.tsv', sep='\t', index=False
)
dev_export.to_csv(
    OUTPUT_DIR / 'cc_v3_dev_predictions.tsv', sep='\t', index=False
)
if test_export is not None:
    test_export.to_csv(
        OUTPUT_DIR / 'cc_v3_test_predictions.tsv', sep='\t', index=False
    )

display(dev_export.head())
print('Saved train OOF:', OUTPUT_DIR / 'cc_v3_train_oof_predictions.tsv')
print('Saved dev:', OUTPUT_DIR / 'cc_v3_dev_predictions.tsv')
print(
    'Saved test:',
    OUTPUT_DIR / 'cc_v3_test_predictions.tsv'
    if test_export is not None else 'pending'
)


In [ ]:
def normalized_rank_error(gold, predicted):
    n = len(gold)
    if n < 2:
        return np.zeros(n)
    gold_rank = rankdata(gold, method='average') / n
    predicted_rank = rankdata(predicted, method='average') / n
    return np.abs(predicted_rank - gold_rank)

diagnostics = dev_df[[
    'id', 'base_id', 'caption_version', TARGET_COLUMN,
    'ref_image_path', 'generated_image_path'
]].rename(columns={TARGET_COLUMN: GOLD_COLUMN}).copy()
diagnostics = diagnostics.merge(
    dev_export.drop(columns=['base_id', 'caption_version']),
    on='id', how='left', validate='one_to_one'
)

diagnostic_prediction_column = (
    'final_prediction' if 'final_prediction' in diagnostics.columns
    else 'raw_prediction' if 'raw_prediction' in diagnostics.columns
    else None
)
if diagnostic_prediction_column is not None:
    diagnostics['diagnostic_prediction'] = diagnostics[
        diagnostic_prediction_column
    ]
    diagnostics['absolute_error'] = (
        diagnostics['diagnostic_prediction'] - diagnostics[GOLD_COLUMN]
    ).abs()
    diagnostics['rank_error'] = normalized_rank_error(
        diagnostics[GOLD_COLUMN], diagnostics['diagnostic_prediction']
    )
else:
    diagnostics['diagnostic_prediction'] = np.nan
    diagnostics['absolute_error'] = np.nan
    diagnostics['rank_error'] = np.nan

display(Markdown(
    f'### Largest CC errors ({diagnostic_prediction_column or "pending"})'
))
display(diagnostics.sort_values('absolute_error', ascending=False).head(20).round(4))

diagnostics.to_csv(
    OUTPUT_DIR / 'cc_v3_per_example_diagnostics.tsv', sep='\t', index=False
)


In [ ]:
def show_diagnostic_cases(frame, count=8):
    if frame['absolute_error'].isna().all():
        print('Diagnostics pending complete dev predictions.')
        return
    selected = frame.sort_values('absolute_error', ascending=False).head(count)
    record_by_id = {
        str(record['instance_id']): record['response']
        for record in dev_cc_records.to_dict('records')
    } if not dev_cc_records.empty else {}
    for _, row in selected.iterrows():
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
        axes[0].imshow(plt.imread(row['ref_image_path']))
        axes[0].set_title('Qatari reference context')
        axes[1].imshow(plt.imread(row['generated_image_path']))
        axes[1].set_title('Generated candidate')
        for axis in axes:
            axis.axis('off')
        fig.suptitle(
            f"{row['id']} | gold={row[GOLD_COLUMN]:.3f} | "
            f"prediction={row['diagnostic_prediction']:.3f} | "
            f"error={row['absolute_error']:.3f}"
        )
        plt.tight_layout()
        plt.show()
        response = record_by_id.get(str(row['id']))
        if response is not None:
            print('Scene category:', response['scene_category'])
            print('Qatar context:', response['qatar_context_status'])
            print('Band probabilities:', response['score_probabilities'])
            print('Expected raw CC:', round(response['raw_cc'], 4))
            print(response['brief_reason'])

show_diagnostic_cases(diagnostics, count=8)


In [ ]:
def conclusion_text():
    lines = ['### Evidence-based Qatari CC conclusion', '']
    lines.append(
        f'- Mapping audit `{MAPPING_AUDIT_VERSION}` repairs train groups '
        'img_017--img_019 and excludes unresolved img_020.'
    )
    lines.append(
        '- Qatar is the only target culture; no Gulf/Arab compatibility hierarchy '
        'is used.'
    )
    lines.append(
        f'- Six train demonstrations use base groups '
        f'{", ".join(sorted(CC_DEMONSTRATION_BASE_IDS))}; those groups are excluded '
        'from train judge evaluation and blend selection.'
    )
    if dev_prediction is None:
        lines.extend([
            '- Dev pilot is pending. Enable `RUN_NEW_API_CALLS=True` while keeping '
            '`PILOT_DEV_ONLY=True`.',
            '- This first stage makes 40 dev calls and no train or test calls.',
        ])
    else:
        raw_row = comparison_table.loc[
            comparison_table['system'].eq('Raw Qatari five-band judge')
        ].iloc[0]
        lines.append(
            f"- Raw five-band dev Spearman: {raw_row['dev_spearman']:.3f}; "
            f"pilot threshold: > {RAW_PILOT_TARGET:.3f}."
        )
        lines.append(f'- Pilot decision: **{PILOT_DECISION}**.')
    if blend_result is None:
        lines.append(
            '- No submission system is frozen until the non-demonstration train '
            'cache and nested grouped CV are complete.'
        )
    else:
        lines.append(
            f'- Train-CV-selected system: **{RECOMMENDED_SYSTEM}**; final raw '
            f'blend weight: {blend_result["final_weight"]:.2f}.'
        )
    lines.extend([
        '- The judge returns only category, Qatar-context status, five score-band '
        'probabilities, and one visible-evidence reason.',
        '- The only calibrated candidate is a one-parameter blend with the '
        'caption-version prior.',
        '- Test calls should remain disabled until train-CV selection is frozen.',
    ])
    return '\n'.join(lines)

display(Markdown(conclusion_text()))
display(comparison_table.round(4))
print('Saved comparison:', OUTPUT_DIR / 'cc_v3_system_comparison.tsv')
print('Saved diagnostics:', OUTPUT_DIR / 'cc_v3_per_example_diagnostics.tsv')
